# Prepare spatially isolated grain-size cross-validation folds

Five stratified group outer folds evaluate each annotated cell once. Three group inner folds select parameters and refit epochs. Same-survey cells whose maximum-radius contexts overlap belong to the same connected spatial group. For maximum radius 3, different partitions must have centre distance greater than 6 cells in the Chebyshev metric. Groups include singleton cells and can contain multiple classes.

The same maximum-radius buffer applies to every grid configuration, all five backbones, all four ensemble rules, and the independent all-data deployment search. Each training/evaluation partition must contain all three classes. Grouping avoids dropping annotations merely to remove overlap; buffer checks still run before fitting. Only metadata is saved, not matrices. Historical outputs require rerunning.

In [1]:
from pathlib import Path
import sys
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor_ss.csv"
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

OUTER_FOLDS = 5
INNER_FOLDS = 3
MAX_CONTEXT_RADIUS = 3
RANDOM_SEED = 42

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from Source.cross_validation import create_cv_plan, save_cv_plan, read_cv_plan

print(f"Project root:     {PROJECT_ROOT}")
print(f"Descriptor:       {DESCRIPTOR_PATH}")
print(f"Cell data root:   {CELL_DATA_ROOT}")
print(f"Output directory: {PREPROCESSED_DIR}")

Project root:     /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach
Descriptor:       /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/Descriptor_ss.csv
Cell data root:   /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/beach_all
Output directory: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed


In [3]:
SOURCE_COLUMNS = {
    "Site": "site",
    "Date": "date",
    "Cell": "cell_id",
    "d50": "d50",
    "Class": "class",
}
DESCRIPTOR_COLUMNS = ["site", "date", "cell_id", "d50", "class"]
DESCRIPTOR_DTYPES = {column: "string" for column in DESCRIPTOR_COLUMNS}


def read_descriptor(path):
    if not path.exists():
        raise FileNotFoundError(f"Descriptor file not found: {path}")
    frame = pd.read_csv(path, dtype="string", encoding="utf-8-sig")
    missing_columns = set(SOURCE_COLUMNS) - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Descriptor is missing columns: {sorted(missing_columns)}")
    frame = frame.rename(columns=SOURCE_COLUMNS)[DESCRIPTOR_COLUMNS]

    frame = frame.dropna(subset=DESCRIPTOR_COLUMNS).copy()
    for column in DESCRIPTOR_COLUMNS:
        frame[column] = frame[column].str.strip()
    empty_mask = frame[DESCRIPTOR_COLUMNS].eq("").any(axis=1)
    if empty_mask.any():
        raise ValueError(f"Empty descriptor values found in {path}: {int(empty_mask.sum())} rows")
    return frame


descriptor_raw = read_descriptor(DESCRIPTOR_PATH)
expected_classes = {"11", "12", "13"}
actual_classes = set(descriptor_raw["class"])
if actual_classes != expected_classes:
    raise ValueError(
        f"Expected second-stage classes {sorted(expected_classes)}, got {sorted(actual_classes)}"
    )

duplicate_mask = descriptor_raw.duplicated(subset=["date", "cell_id"], keep=False)
if duplicate_mask.any():
    duplicates = descriptor_raw.loc[duplicate_mask].sort_values(["date", "cell_id", "class"])
    conflicting_keys = (
        duplicates.groupby(["date", "cell_id"])["class"].nunique().loc[lambda counts: counts > 1]
    )
    if not conflicting_keys.empty:
        conflicts = duplicates.merge(
            conflicting_keys.rename("class_count"),
            on=["date", "cell_id"],
        )
        display(conflicts)
        raise ValueError("Conflicting classes found for duplicate date/cell_id pairs.")
    duplicate_key_count = duplicates[["date", "cell_id"]].drop_duplicates().shape[0]
    print(f"Dropping {len(duplicates) - duplicate_key_count:,} repeated annotations.")

descriptor = (
    descriptor_raw
    .drop_duplicates(subset=["date", "cell_id"], keep="first")
    .sort_values(["date", "cell_id"])
    .reset_index(drop=True)
)
descriptor.head(), descriptor.shape

(  site      date  cell_id   d50 class
 0   67  20220826  1000972  40.1    13
 1   68  20220826  1031549  20.9    12
 2   69  20220826  1045306  17.5    12
 3   70  20220826  1069762  23.7    12
 4   71  20220826  1094222  34.5    13,
 (140, 5))

In [4]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Cell data root not found: {CELL_DATA_ROOT}")

missing_rows = []
for row in descriptor.to_dict(orient="records"):
    missing_features = [
        suffix
        for suffix in FEATURE_SUFFIXES
        if not (CELL_DATA_ROOT / row["date"] / f"{row['cell_id']}{suffix}.txt").is_file()
    ]
    if missing_features:
        missing_rows.append(
            {
                "date": row["date"],
                "cell_id": row["cell_id"],
                "class": row["class"],
                "missing_features": ",".join(missing_features),
            }
        )

missing_cells = pd.DataFrame(missing_rows)
if not missing_cells.empty:
    display(missing_cells.head(50))
    raise FileNotFoundError(
        f"{len(missing_cells):,} annotated cells are incomplete under {CELL_DATA_ROOT}."
    )

print(f"Validated five feature files for all {len(descriptor):,} annotated cells in {CELL_DATA_ROOT}.")

class_counts = descriptor["class"].value_counts().sort_index()
class_counts.to_frame(name="count")

Validated five feature files for all 140 annotated cells in /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/beach_all.


,count
class,
11,22
12,64
13,54


In [5]:
samples, inner_folds = create_cv_plan(
    descriptor, outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS,
    random_seed=RANDOM_SEED, max_context_radius=MAX_CONTEXT_RADIUS,
)
display(samples.groupby(["outer_fold", "class"]).size().unstack(fill_value=0))
print(f"Spatial groups: {samples.spatial_group.nunique()}; maximum context radius: {MAX_CONTEXT_RADIUS}", flush=True)

,count
split,
train,98
validation,21
test,21


In [6]:
assert len(samples) == len(descriptor)
assert not samples.duplicated(["date", "cell_id"]).any()
assert samples.groupby("spatial_group").outer_fold.nunique().eq(1).all()
for outer in range(OUTER_FOLDS + 1):
    assignment = inner_folds.loc[inner_folds.outer_fold == outer]
    assert assignment.groupby("spatial_group").inner_fold.nunique().eq(1).all()
print("Spatial groups are intact in outer and inner folds; full geometric validation runs when loading the plan.", flush=True)

class,11,12,13
split,,,
train,15,45,38
validation,3,10,8
test,4,9,8


In [7]:
manifest = save_cv_plan(samples, inner_folds, PREPROCESSED_DIR, random_seed=RANDOM_SEED)
print("Saved gs_cv_samples.csv, gs_cv_inner_folds.csv, gs_cv_manifest.json", flush=True)
print("Legacy gs_*_split.csv files are historical and are not used for training/evaluation.", flush=True)
display(pd.Series(manifest).to_frame("value"))
read_cv_plan(PREPROCESSED_DIR)
print("Verified all classes and zero maximum-radius context overlap across all fit/evaluation boundaries.", flush=True)

       all: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_dataset_splits.csv
     train: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_train_split.csv
validation: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_validation_split.csv
      test: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_test_split.csv
